# Bharat Ledger — NVIDIA cudf.pandas Acceleration Benchmark

**Run this notebook on Google Colab with a free T4 GPU runtime**
(`Runtime` -> `Change runtime type` -> `T4 GPU`). This is mandatory: RAPIDS/cudf requires an
NVIDIA GPU + CUDA and cannot run on the local macOS development machine used for the rest of
this project (PLAN.md Phase 4).

**What this notebook does:**
1. Loads the large (~6M row) synthetic district-level table generated by
   `pipeline/generate_synthetic.py` (`data/synthetic/district_flat_large.parquet`).
2. Runs the identical state x sector x year aggregation (median cost/unit -> rank ->
   normalize to an efficiency score) in (a) stock pandas and (b) `cudf.pandas`.
3. Times both, prints the speedup, and verifies the two engines produce **identical**
   rankings (correctness, not just speed).
4. Exports the resulting `rankings` table as CSV, matching `pipeline/schema.py`'s
   `RankingRow` shape, so it can be imported back into the app via
   `pipeline.build_db.import_rankings_csv()` — this becomes the *authoritative* rankings
   table, superseding the CPU-pandas fallback the app ships with by default.

**Before running:** upload `district_flat_large.parquet` (from `data/synthetic/` in this
repo) using the file-upload cell below, or mount Google Drive if you've placed it there.

In [ ]:
# Confirm we actually have a GPU runtime before going further.
!nvidia-smi

In [ ]:
# Install RAPIDS cudf via the official rapidsai Colab install helper.
# This installs the cudf build matched to Colab's current CUDA/driver version.
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

In [ ]:
# Upload district_flat_large.parquet from your local data/synthetic/ directory.
# (Alternative: mount Google Drive and point PARQUET_PATH at the file there instead.)
from google.colab import files
uploaded = files.upload()  # select data/synthetic/district_flat_large.parquet
PARQUET_PATH = list(uploaded.keys())[0]
print('Using:', PARQUET_PATH)

In [ ]:
import time
import pandas as pd

def run_aggregation(df):
    """Identical aggregation logic run on both engines: state x sector x year median
    cost-per-unit, then rank + normalize to a 0-100 efficiency score within each
    (country, sector, year) group. Mirrors pipeline/build_db.py's
    build_fallback_rankings() CPU logic exactly, so results are directly comparable."""
    grouped = (
        df.groupby(['country', 'sector', 'year', 'state'])['cost_per_unit_cr']
        .median()
        .reset_index()
        .rename(columns={'cost_per_unit_cr': 'median_cost_per_unit_cr'})
    )

    out_frames = []
    for (country, sector, year), group in grouped.groupby(['country', 'sector', 'year']):
        group = group.sort_values('median_cost_per_unit_cr').reset_index(drop=True)
        lo = group['median_cost_per_unit_cr'].min()
        hi = group['median_cost_per_unit_cr'].max()
        span = max(hi - lo, 1e-9)
        group['rank'] = range(1, len(group) + 1)
        group['efficiency_score'] = (100 * (1 - (group['median_cost_per_unit_cr'] - lo) / span)).round().astype(int)
        out_frames.append(group)

    return pd.concat(out_frames, ignore_index=True)

## Baseline: stock pandas

In [ ]:
df_cpu = pd.read_parquet(PARQUET_PATH)
print(f'Loaded {len(df_cpu):,} rows')

t0 = time.perf_counter()
rankings_cpu = run_aggregation(df_cpu)
cpu_seconds = time.perf_counter() - t0
print(f'Stock pandas aggregation: {cpu_seconds:.3f}s -> {len(rankings_cpu)} ranking rows')

## Accelerated: cudf.pandas (GPU)

`%load_ext cudf.pandas` must be the FIRST cudf-related thing in a fresh cell/kernel context
for the zero-code-change accelerator to patch pandas correctly — we re-import pandas after
loading the extension, per NVIDIA's documented usage.

In [ ]:
%load_ext cudf.pandas
import pandas as pd  # re-import AFTER loading the extension -> this pandas is now GPU-backed

df_gpu = pd.read_parquet(PARQUET_PATH)

t0 = time.perf_counter()
rankings_gpu = run_aggregation(df_gpu)
gpu_seconds = time.perf_counter() - t0
print(f'cudf.pandas aggregation: {gpu_seconds:.3f}s -> {len(rankings_gpu)} ranking rows')

## Results: speedup + correctness check

In [ ]:
speedup = cpu_seconds / gpu_seconds
print(f'Rows aggregated: {len(df_cpu):,}')
print(f'Stock pandas:    {cpu_seconds:.3f}s')
print(f'cudf.pandas:     {gpu_seconds:.3f}s')
print(f'Speedup:         {speedup:.1f}x')

# Correctness: confirm both engines agree on the actual rankings, not just similar timing.
merged = rankings_cpu.merge(
    rankings_gpu.to_pandas() if hasattr(rankings_gpu, 'to_pandas') else rankings_gpu,
    on=['country', 'sector', 'year', 'state'], suffixes=('_cpu', '_gpu'),
)
rank_matches = (merged['rank_cpu'] == merged['rank_gpu']).mean()
print(f'Rank agreement between engines: {rank_matches * 100:.1f}% of rows')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Stock pandas\n(CPU)', 'cudf.pandas\n(NVIDIA T4 GPU)'], [cpu_seconds, gpu_seconds],
              color=['#8A8F99', '#0E8F8F'])
ax.set_ylabel('Aggregation time (seconds)')
ax.set_title(f'Bharat Ledger state-ranking aggregation\n{len(df_cpu):,} rows — {speedup:.1f}x speedup on GPU')
for bar, val in zip(bars, [cpu_seconds, gpu_seconds]):
    ax.text(bar.get_x() + bar.get_width() / 2, val, f'{val:.2f}s', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('cudf_speedup_chart.png', dpi=150)
plt.show()
print('Saved cudf_speedup_chart.png — use this in the pitch deck acceleration slide.')

## Export the authoritative rankings table

Download `rankings_gpu.csv` and replace `data/synthetic/` locally, then run:
```bash
python3 -c "from pipeline.build_db import import_rankings_csv; import_rankings_csv('rankings_gpu.csv')"
```
to load these GPU-computed rankings into the app's SQLite store, superseding the CPU fallback.

In [ ]:
export_df = rankings_gpu.to_pandas() if hasattr(rankings_gpu, 'to_pandas') else rankings_gpu
export_df = export_df[['country', 'sector', 'year', 'state', 'median_cost_per_unit_cr', 'rank', 'efficiency_score']]
export_df.to_csv('rankings_gpu.csv', index=False)

from google.colab import files
files.download('rankings_gpu.csv')
files.download('cudf_speedup_chart.png')
print('Downloaded rankings_gpu.csv and cudf_speedup_chart.png')